# pyfixest regression + MLflow tracking

A short end-to-end example: make some (polars) data, run a stepwise sweep of regressions through `regress` (which wraps a pyfixest model and logs it to MLflow), then pull the logged runs back with `coeftable` and `etable`.

In [ ]:
import warnings

# keep outputs reproducible: this warning embeds an absolute path
warnings.filterwarnings("ignore", message="IProgress not found")

import logging
import os

# artifact-download progress bars print throughput rates -> not reproducible
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

import mlflow
import numpy as np
import polars as pl

from pyfixest_regression import coeftable, etable, regress

# mlflow's INFO logs carry timestamps; silence them so outputs are stable
logging.getLogger("mlflow").setLevel(logging.ERROR)

mlflow.set_tracking_uri("sqlite:///mlflow.db")
_ = mlflow.set_experiment("pyfixest-regression-example")

## Make some data

A treatment-effect DGP as a **polars** DataFrame (`regress` is dataframe-agnostic via narwhals — pandas works too). `age` and `income` are drawn **correlated** (cov ≈ 0.6), so dropping one shifts the other's coefficient — which makes comparing specifications interesting.

`outcome = 1 + 2·treat − 0.03·age + 0.5·income + noise`

In [ ]:
rng = np.random.default_rng(0)
n = 500

treat = rng.integers(0, 2, size=n)  # binary treatment (0/1)
# age and income are correlated (cov 6 over sds 10 and 1 -> corr ~ 0.6)
age, income = rng.multivariate_normal(
    mean=[50.0, 0.0], cov=[[100.0, 6.0], [6.0, 1.0]], size=n
).T
outcome = 1.0 + 2.0 * treat - 0.03 * age + 0.5 * income + rng.normal(size=n)

data = pl.DataFrame({"outcome": outcome, "treat": treat, "age": age, "income": income})
data.head()

outcome,treat,age,income
f64,i64,f64,f64
0.983391,1,48.02623,-0.424792
1.259387,1,24.491673,-1.799903
4.325008,1,62.202373,0.898371
0.961788,0,50.33702,0.873421
-0.951984,0,59.177499,1.197965


## Run a stepwise sweep

`csw(...)` (cumulative stepwise) fits a sequence of nested models in one call — here `treat`, then `+ age`, then `+ income`. `regress` logs each resolved model as its own run and returns the list of fits. `dataset_version` records which version of the data a run used (it is what identifies runs for dedup — the data itself is not hashed, so bump it when the data changes).

In [ ]:
fits = regress("outcome ~ csw(treat, age, income)", data=data, dataset_version="v1")

fits[-1].summary()  # the last (full) model

###

Estimation:  OLS
Dep. var.: outcome
sample: None = all
Inference:  iid
Observations:  500

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Intercept     |      1.301 |        0.310 |     4.201 |      0.000 |  0.692 |   1.909 |
| treat         |      1.948 |        0.094 |    20.782 |      0.000 |  1.763 |   2.132 |
| age           |     -0.035 |        0.006 |    -5.895 |      0.000 | -0.046 |  -0.023 |
| income        |      0.478 |        0.060 |     7.963 |      0.000 |  0.360 |   0.596 |
---
RMSE: 1.033 R2: 0.502 


## Every coefficient across the runs

`coeftable` stacks each run's coefficients into one long, self-describing frame (one row per run × coefficient), with the formula and the run's params joined on. It returns **polars** by default (`backend="pandas"` for pandas); columns use plain snake_case names, including the 95% CI (`ci_low`/`ci_high`).

In [ ]:
coefs = coeftable("pyfixest-regression-example")
coefs[["fml", "coefficient", "estimate", "std_error", "p_value", "ci_low", "ci_high"]]

fml,coefficient,estimate,std_error,p_value,ci_low,ci_high
str,str,f64,f64,f64,f64,f64
"""outcome ~ treat + age + income""","""Intercept""",1.300891,0.309679,0.000032,0.692446,1.909336
"""outcome ~ treat + age + income""","""treat""",1.947587,0.093715,0.0,1.763458,2.131715
"""outcome ~ treat + age + income""","""age""",-0.034867,0.005915,6.9000e-9,-0.046488,-0.023245
"""outcome ~ treat + age + income""","""income""",0.478348,0.060069,0.0,0.360327,0.596369
"""outcome ~ treat + age""","""Intercept""",-0.103121,0.270103,0.702785,-0.633806,0.427564
"""outcome ~ treat + age""","""treat""",1.931537,0.099403,0.0,1.736235,2.126839
"""outcome ~ treat + age""","""age""",-0.007105,0.005069,0.161672,-0.017065,0.002855
"""outcome ~ treat""","""Intercept""",-0.467444,0.073461,4.0000e-10,-0.611776,-0.323111
"""outcome ~ treat""","""treat""",1.944681,0.099055,0.0,1.750064,2.139299


## Side-by-side regression table

`etable` puts the runs side by side — one column per model, headed by the run's name or, when unnamed, an abbreviated formula (the `term` column holds the coefficient/stat labels, since polars has no row index). Watch the `age` coefficient shift once the correlated `income` enters.

In [ ]:
etable("pyfixest-regression-example")

term,treat,treat + age,treat + age + income
str,str,str,str
"""Intercept""","""-0.467*** (0.073)""","""-0.103 (0.270)""","""1.301*** (0.310)"""
"""treat""","""1.945*** (0.099)""","""1.932*** (0.099)""","""1.948*** (0.094)"""
"""age""","""""","""-0.007 (0.005)""","""-0.035*** (0.006)"""
"""income""","""""","""""","""0.478*** (0.060)"""
"""fml""","""outcome ~ treat""","""outcome ~ treat + age""","""outcome ~ treat + age + income"""
"""nobs""","""500""","""500""","""500"""
"""r2""","""0.436""","""0.439""","""0.502"""
"""adj_r2""","""0.435""","""0.436""","""0.499"""
